# Dinámica de la Desigualdad Espacial

En esta clase estudiamos cómo medir la desigualdad económica entre regiones y, sobre todo, cómo incorporar el **espacio** en esas mediciones. La pregunta de fondo es:

> *No basta con saber cuánta desigualdad hay; también queremos saber dónde están los ricos y los pobres, y si esa geografía cambia en el tiempo.*

Trabajaremos con datos de ingreso per cápita de los condados de Estados Unidos entre 1969 y 2017. Es un dataset particularmente útil porque combina dos dimensiones que rara vez vemos juntas:

- **Espacial:** más de 3000 condados con geometría conocida.
- **Temporal:** casi 50 años de observaciones para cada condado.

## Lo que veremos

1. Medidas clásicas de desigualdad (no espaciales): ratio 20:20, índice de Gini, índice de Theil.
2. La diferencia entre desigualdad **personal** y desigualdad **regional**.
3. Tres formas de meter el espacio en el análisis:
   - Autocorrelación espacial del ingreso (Moran's I).
   - Descomposición regional del índice de Theil (componentes *entre* y *dentro*).
   - Gini espacial: separar diferencias entre vecinos vs. no vecinos.

Adaptado de Rey, Arribas-Bel & Wolf (2020), *Geographic Data Science with Python*, Capítulo 9.

## 1. Preparación: librerías y datos

In [ ]:
import seaborn
import pandas
import geopandas
import numpy
import esda
import matplotlib.pyplot as plt
from libpysal import weights
from pysal.explore import inequality

In [ ]:
# Cargamos el dataset de ingreso per cápita por condado
pci_df = geopandas.read_file(
    "datos/external/us_county_income/uscountypcincome.gpkg"
)
pci_df.shape

Tenemos 3076 condados (filas) y 77 columnas. Veamos cómo están organizadas las columnas:

In [ ]:
pci_df.columns.tolist()

### Formato wide vs. long

Notemos algo importante: **cada año es una columna**. Esto es lo que se llama formato *wide* (ancho). En contraste, el formato *long* (largo) tendría una fila por cada combinación condado-año.

| Formato | Estructura | Ventaja |
|---------|------------|---------|
| **Wide** | una fila por entidad, cada periodo es columna | bueno para analizar trayectorias |
| **Long** | una fila por combinación entidad-tiempo | bueno para gráficos con seaborn, agregaciones |

Para análisis de trayectorias temporales (que es lo que haremos), wide es más cómodo. Veamos por ejemplo la evolución del ingreso del condado de Jackson, Mississippi en sus primeros 10 años:

In [ ]:
pci_df.query('NAME == "Jackson" & STATEFP == "28"').loc[:, "1969":"1979"]

## 2. Visualización inicial

Antes de calcular índices, miremos la distribución del ingreso. Hay dos visualizaciones complementarias:

- Un **histograma**: muestra la forma de la distribución de valores (la *feature*).
- Un **mapa coroplético**: muestra la distribución *geográfica*.

Las dos cuentan historias distintas. Necesitamos las dos.

In [ ]:
seaborn.histplot(x=pci_df["1969"], kde=True);

La distribución tiene una **cola larga a la derecha** (sesgo positivo). Esto es característico de los ingresos: hay relativamente pocos súper-ricos pero los súper-ricos están *muy* lejos de la media.

**Nota importante:** estamos viendo *ingresos per cápita por condado*, no ingresos individuales. La agregación por condado suaviza la distribución: dentro de cada condado conviven ricos y pobres, pero solo vemos el promedio. Esto significa que nuestras conclusiones serán sobre *condados*, no sobre *personas*. Confundir ambos niveles se llama **falacia ecológica**.

In [ ]:
# Reproyectamos a una proyección equivalente para mapear EE.UU.
pci_df = pci_df.to_crs(epsg=5070)  # Albers Equal Area Norteamérica

In [ ]:
ax = pci_df.plot(
    column="1969",
    scheme="Quantiles",
    legend=True,
    edgecolor="none",
    legend_kwds={"loc": "lower left"},
    figsize=(12, 8),
)
ax.set_axis_off()
plt.show()

El mapa muestra patrones geográficos claros: el sur aparece consistentemente más pobre, las costas más ricas. El histograma no nos decía nada de esto.

## 3. Medidas globales de desigualdad

Ahora calcularemos índices que resumen la desigualdad en un solo número. "Globales" significa que describen la distribución completa, sin importar dónde está cada observación.

### 3.1 Ratio 20:20

Es la medida más simple: el ingreso del percentil 80 dividido por el del percentil 20.

$$ \text{Ratio 20:20} = \frac{P_{80}}{P_{20}} $$

Si vale 1, el rico y el pobre ganan lo mismo. Si vale 5, el rico gana 5 veces lo que el pobre. Su gracia es que ignora los extremos (super ricos y super pobres), así que es robusta a outliers.

In [ ]:
top20, bottom20 = pci_df["1969"].quantile([0.8, 0.2])
ratio_1969 = top20 / bottom20
print(f"En 1969 el percentil 80 ganaba {ratio_1969:.2f} veces lo del percentil 20")

Para ver cómo evoluciona en el tiempo, definimos una función y la aplicamos a cada año:

In [ ]:
def ratio_20_20(valores):
    p80, p20 = valores.quantile([0.8, 0.2])
    return p80 / p20

# Lista de años como strings (porque así están las columnas)
anios = numpy.arange(1969, 2018).astype(str)

# Aplicamos la función a cada columna de año
ratio_2020 = pci_df[anios].apply(ratio_20_20, axis=0)

**Sobre el patrón `.apply()`:** lo vamos a usar mucho. La idea es: tenemos una función que sabe procesar *un* año (una columna), y `apply` la corre sobre *todas* las columnas de años. Es equivalente a un `for` pero más compacto.

El equivalente con loop sería:
```python
ratio_2020 = {}
for anio in anios:
    ratio_2020[anio] = ratio_20_20(pci_df[anio])
ratio_2020 = pandas.Series(ratio_2020)
```

Las dos formas dan lo mismo. Usen la que les resulte más clara.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(anios, ratio_2020)
ax.set_xticks(anios[::5])
ax.set_xlabel("Año")
ax.set_ylabel("Ratio 20:20")
ax.set_title("Evolución de la desigualdad entre condados (Ratio 20:20)")
plt.xticks(rotation=45)
plt.show()

El patrón tiene forma de U: la desigualdad cae hasta mediados de los 90, sube hasta 2013, y luego empieza a bajar de nuevo.

### 3.2 Índice de Gini

El Gini es probablemente el índice de desigualdad más conocido. Su construcción se basa en la **curva de Lorenz**, que grafica:

- En el eje X: el porcentaje acumulado de la población (ordenada de menor a mayor ingreso).
- En el eje Y: el porcentaje acumulado del ingreso total que poseen.

Si la sociedad fuera perfectamente igualitaria, el 50% más pobre tendría exactamente el 50% del ingreso, y el gráfico sería la diagonal. Cuanto más se aleja la curva real de la diagonal, más desigualdad hay.

**El Gini es exactamente eso: el área entre la curva de Lorenz y la línea de igualdad, normalizada.** Va de 0 (igualdad perfecta) a 1 (un solo individuo tiene todo).

In [ ]:
def lorenz(y):
    """Devuelve la curva de Lorenz: (proporción de población, proporción de ingreso acumulada)."""
    y = numpy.asarray(y)
    ingresos_ordenados = numpy.sort(y)
    proporcion_ingreso = (ingresos_ordenados / ingresos_ordenados.sum()).cumsum()
    n = len(y)
    proporcion_poblacion = numpy.arange(1, n + 1) / n
    return proporcion_poblacion, proporcion_ingreso

In [ ]:
pob_1969, ing_1969 = lorenz(pci_df["1969"])

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(pob_1969, ing_1969, label="Curva de Lorenz (1969)", linewidth=2)
ax.plot([0, 1], [0, 1], color="red", label="Igualdad perfecta", linestyle="--")
ax.fill_between(pob_1969, pob_1969, ing_1969, alpha=0.2)
ax.set_xlabel("Proporción acumulada de condados")
ax.set_ylabel("Proporción acumulada de ingreso")
ax.set_title("Curva de Lorenz, ingresos por condado (1969)")
ax.legend()
plt.show()

El área sombreada entre las dos curvas es lo que captura el Gini.

Para calcular el Gini directamente, usamos la clase `Gini` de pysal:

In [ ]:
g69 = inequality.gini.Gini(pci_df["1969"].values)
print(f"Gini 1969: {g69.g:.4f}")

Un valor de 0.135 es bajo. Atención: este es el Gini *entre condados*, no *entre personas*. El Gini de ingresos personales en EE.UU. está cerca de 0.4. Ya sabíamos que la agregación oculta variabilidad.

Calculemos el Gini para todos los años:

In [ ]:
def gini_de_columna(columna):
    return inequality.gini.Gini(columna.values).g

desigualdades = pci_df[anios].apply(gini_de_columna, axis=0).to_frame("gini")
desigualdades.head()

In [ ]:
desigualdades["gini"].plot(figsize=(10, 4), title="Evolución del Gini entre condados");

El patrón es similar al del ratio 20:20, lo que es consistente: ambos miden lo mismo desde ángulos distintos.

### 3.3 Índice de Theil

El tercer índice clásico es el de Theil. Conceptualmente está relacionado con la entropía: mide qué tan "uniformemente" se reparte el ingreso. Es 0 cuando todos tienen lo mismo y crece a medida que aparece más concentración.

Su gran ventaja, que aprovecharemos más adelante, es que es **descomponible**: se puede separar en componentes entre regiones y dentro de regiones. El Gini no permite esto de manera tan limpia.

In [ ]:
def theil_de_columna(columna):
    return inequality.theil.Theil(columna.values).T

desigualdades["theil"] = pci_df[anios].apply(theil_de_columna, axis=0)

desigualdades["theil"].plot(
    figsize=(10, 4), color="orange", title="Evolución del Theil entre condados"
);

### ¿Para qué tener tres índices si dan la misma forma?

A primera vista parecen redundantes. Pero no lo son del todo:

In [ ]:
seaborn.regplot(x="theil", y="gini", data=desigualdades);

Están correlacionados pero no son perfectamente equivalentes. Cada uno tiene propiedades distintas:

- **Ratio 20:20** es robusto a outliers pero ignora todo lo que pasa en los extremos.
- **Gini** es sensible a cambios cerca de la mediana y permite descomposición espacial (como veremos).
- **Theil** se descompone naturalmente en partes "entre grupos" y "dentro de grupos".

En la práctica suele ser bueno reportar al menos dos para asegurarse de que las conclusiones no dependen del índice elegido.

## 4. Desigualdad personal vs. desigualdad regional

Hasta acá medimos desigualdad **entre condados**. Pero hay una distinción importante que se confunde con frecuencia.

Imaginemos un país hipotético con dos regiones:

- **Región A:** todos ganan exactamente $1000.
- **Región B:** la mitad gana $500, la otra mitad gana $1500. El promedio también es $1000.

**Desigualdad regional:** cero. Las dos regiones tienen el mismo ingreso promedio.

**Desigualdad personal:** alta. Dentro de B hay diferencias de 3 a 1.

Cuando trabajamos con datos agregados (como el ingreso *promedio* por condado), estamos asumiendo implícitamente que **dentro de cada condado todos ganan el promedio**. Es un supuesto fuerte, pero permite aislar la pregunta: *¿cuánta desigualdad existe entre lugares, independiente de la que hay dentro de cada lugar?*

Esto es lo que en el capítulo se llama "shift one level up the spatial hierarchy": subir de la persona al lugar.

Para responder preguntas sobre desigualdad personal necesitaríamos microdatos (encuestas tipo CASEN, ACS individual, etc.), no datos agregados como los que tenemos.

## 5. Desigualdad espacial

Acá llegamos a lo que distingue al análisis espacial del análisis económico tradicional. Los índices anteriores (Gini, Theil, ratio) **ignoran completamente la geografía**: si tomáramos los condados y los ordenáramos al azar en el mapa, los índices serían idénticos. Pero la geografía importa.

Veremos tres formas de incorporarla:

1. **Autocorrelación espacial:** ¿los condados ricos están cerca de otros condados ricos?
2. **Descomposición regional:** ¿cuánta de la desigualdad total viene de diferencias entre regiones (ej. Sur vs. Noreste) y cuánta de diferencias dentro de cada región?
3. **Gini espacial:** ¿las diferencias de ingreso son entre vecinos o entre lugares lejanos?

### 5.1 Autocorrelación espacial: Moran's I del ingreso

Calculamos el Moran's I del ingreso para cada año. Si es positivo y significativo, hay clusters espaciales: condados ricos junto a condados ricos, condados pobres junto a condados pobres.

In [ ]:
# Matriz de pesos por contigüidad tipo Queen
wq = weights.Queen.from_dataframe(pci_df)

In [ ]:
def moran_de_columna(y, w=wq):
    mo = esda.Moran(y, w=w)
    return pandas.Series({"I": mo.I, "p_valor": mo.p_sim})

moran_stats = pci_df[anios].apply(moran_de_columna, axis=0).T
moran_stats.head()

In [ ]:
# Agregamos los resultados a nuestra tabla principal
desigualdades = desigualdades.join(moran_stats)

desigualdades[["I", "p_valor"]].plot(subplots=True, figsize=(10, 6))
plt.show()

Hay dos cosas notables:

1. **Moran's I baja consistentemente en el tiempo.** Mientras los Gini/Theil tienen forma de U, Moran sigue cayendo. Esto significa que la **estructura geográfica** de la desigualdad se está debilitando: hay desigualdad, pero está cada vez menos concentrada espacialmente.
2. **El p-valor siempre es 0.001** (significativo). Aunque la autocorrelación cae, nunca deja de ser estadísticamente significativa.

**Lección importante:** la desigualdad agregada (Gini, Theil) y la estructura espacial de la desigualdad (Moran) son cosas distintas. Pueden moverse en direcciones distintas. El análisis espacial agrega información que los índices clásicos no capturan.

### 5.2 Descomposición regional del Theil

Una crítica habitual a los índices globales es que esconden el origen de la desigualdad. ¿Es porque hay diferencias *entre regiones* (el Sur vs. el Noreste)? ¿O porque dentro de cada región conviven condados muy distintos?

El Theil permite separar estos dos componentes:

$$ T = T_{\text{entre}} + T_{\text{dentro}} $$

- $T_{\text{entre}}$: cuánta desigualdad existe si reemplazamos cada condado por el promedio de su región. Mide diferencias entre regiones.
- $T_{\text{dentro}}$: cuánta desigualdad queda dentro de cada región, después de descontar las diferencias entre regiones.

Los datos vienen con una clasificación en 8 regiones censales:

In [ ]:
nombres_regiones = {
    1: "New England",
    2: "Mideast",
    3: "Great Lakes",
    4: "Plains",
    5: "Southeast",
    6: "Southwest",
    7: "Rocky Mountain",
    8: "Far West",
}

In [ ]:
ax = pci_df.assign(
    Region_Name=pci_df.Region.map(nombres_regiones)
).plot(
    "Region_Name",
    linewidth=0,
    legend=True,
    categorical=True,
    legend_kwds=dict(bbox_to_anchor=(1.2, 0.5)),
    figsize=(12, 8),
)
ax.set_axis_off();

Veamos cómo evoluciona el ingreso promedio en cada región:

In [ ]:
promedios_region = (
    pci_df.assign(Region_Name=pci_df.Region.map(nombres_regiones))
    .groupby("Region_Name")[anios]
    .mean()
)

promedios_region.T.plot.line(figsize=(10, 5), title="Ingreso promedio por región")
plt.ylabel("Ingreso per cápita promedio");

Ahora calculamos la descomposición de Theil con `TheilD`:

In [ ]:
theil_dr = inequality.theil.TheilD(pci_df[anios].values, pci_df.Region)

El objeto resultante tiene dos atributos clave:
- `.bg` (between groups): componente entre regiones para cada año.
- `.wg` (within groups): componente dentro de regiones para cada año.

In [ ]:
desigualdades["theil_entre"] = theil_dr.bg
desigualdades["theil_dentro"] = theil_dr.wg
desigualdades["theil_entre_proporcion"] = (
    desigualdades["theil_entre"] / desigualdades["theil"]
)

In [ ]:
desigualdades[
    ["theil_entre", "theil_dentro", "theil_entre_proporcion"]
].plot(subplots=True, figsize=(10, 8))
plt.show()

**Cómo leer este gráfico:**

- El componente *entre* regiones cae fuertemente entre 1969 y los 90 y luego se mantiene relativamente estable. Esto refleja la convergencia de regiones tradicionalmente pobres (como el Sur) hacia el promedio nacional.
- El componente *dentro* de las regiones sube en los últimos años: la desigualdad reciente viene cada vez más de diferencias entre condados de la *misma* región.
- La proporción que aporta lo *entre* regiones (panel inferior) toca su mínimo en los 2000, no en los 90 como el Gini total. Conclusión: la desigualdad regional clásica importa cada vez menos para explicar el total.

Esta es información que **los índices globales no podían darnos**.

### 5.3 Gini espacial: vecinos vs. no vecinos

La descomposición regional asume que las regiones están dadas (Sur, Noreste, etc.). Pero ¿qué pasa si queremos algo más fino, basado en vecindad real?

El **Gini espacial** descompone el Gini estándar en dos partes:

- Diferencias entre **pares de vecinos**.
- Diferencias entre **pares de no vecinos**.

La intuición: si hay fuerte autocorrelación espacial positiva (vecinos parecidos), la mayor parte del Gini debería venir de las diferencias entre lugares lejanos. Si la geografía no importa, ambos componentes deberían ser similares (proporcionalmente al número de pares).

El test de inferencia compara el componente observado contra lo que se obtendría si los ingresos se distribuyeran al azar en el mapa.

In [ ]:
from inequality.gini import Gini_Spatial

# El Gini espacial requiere matriz binaria (0/1, no estandarizada por filas)
wq.transform = "B"

In [ ]:
gs69 = Gini_Spatial(pci_df["1969"], wq)

print(f"Gini total 1969: {gs69.g:.4f}")
print(f"Componente entre no-vecinos: {gs69.wcg_share:.4f}")
print(f"P-valor: {gs69.p_sim}")

Casi todo el Gini (0.1354 de 0.1356) viene de diferencias entre **no vecinos**. La fracción aportada por los vecinos es minúscula. Esto refleja autocorrelación positiva fuerte: los vecinos se parecen, así que las diferencias grandes ocurren entre lugares distantes.

**Cuidado al interpretar:** el componente "entre vecinos" es pequeño en parte porque hay pocos pares de vecinos comparados con todos los pares posibles:

In [ ]:
print(f"Pares de vecinos: {wq.pct_nonzero:.2f}% del total de pares posibles")

Solo el 0.19% de los pares posibles son vecinos. La inferencia (p-valor) es lo importante: nos dice si lo observado es distinto de lo que pasaría con ingresos aleatoriamente distribuidos en el mapa.

Calcular el Gini espacial para todos los años toma alrededor de un minuto:

In [ ]:
def gini_espacial_de_columna(ingresos, pesos):
    gs = Gini_Spatial(ingresos, pesos)
    denom = 2 * ingresos.mean() * pesos.n ** 2
    return pandas.Series({
        "gini": gs.g,
        "diferencias_cercanas": gs.wg / denom,
        "diferencias_lejanas": gs.wcg / denom,
        "p_valor": gs.p_sim,
    })

In [ ]:
%%time
gini_espacial = pci_df[anios].apply(gini_espacial_de_columna, pesos=wq).T
gini_espacial.head()

In [ ]:
desigualdades["diferencias_cercanas"] = gini_espacial["diferencias_cercanas"]

desigualdades[["diferencias_cercanas", "I"]].plot.line(
    subplots=True, figsize=(12, 6)
)
plt.show()

**Relación entre los dos paneles:** las diferencias entre vecinos suben con el tiempo, mientras Moran's I baja. Esto es coherente: a medida que la autocorrelación espacial se debilita, los vecinos se parecen menos, así que las diferencias entre ellos crecen. Son dos formas distintas de capturar la misma historia.

## 6. Conclusión

Las medidas clásicas de desigualdad (Gini, Theil, ratio 20:20) tratan a las observaciones como si fueran intercambiables. Para datos espaciales esto pierde información valiosa: dos sociedades pueden tener el mismo Gini pero geografías de la desigualdad muy distintas.

Vimos tres formas complementarias de incorporar el espacio:

1. **Moran's I del ingreso:** mide qué tan agrupados están geográficamente los ricos y los pobres.
2. **Descomposición regional del Theil:** separa cuánta desigualdad viene de diferencias entre regiones predefinidas vs. dentro de ellas.
3. **Gini espacial:** separa diferencias entre vecinos vs. no vecinos, sin necesidad de definir regiones.

**Hallazgo clave del caso de EE.UU.:** entre 1969 y 2017, la desigualdad agregada tuvo forma de U, pero la *estructura geográfica* de la desigualdad cayó de forma sostenida. El sur dejó de ser sistemáticamente más pobre. Hoy la desigualdad ocurre más bien dentro de cada región que entre regiones. Sin las herramientas espaciales, esta historia no aparece.

## Ejercicios

1. Repliquen el ratio 20:20 usando los percentiles 10 y 90 en vez de 20 y 80. ¿Cambia mucho la trayectoria? ¿Qué les dice eso?
2. Calculen el Gini para 1969 y para 2017 y comparen las curvas de Lorenz superpuestas. ¿Visualmente qué cambió?
3. La descomposición de Theil dependió de las 8 regiones censales. ¿Qué pasa si usan en cambio los estados (50 grupos)? Compare la proporción `theil_entre / theil_total`.
4. **Aplicación a Chile:** consigan datos de ingreso autónomo por comuna (CASEN) o de remuneración promedio por comuna (SII) para distintos años. Replique el análisis. ¿La geografía de la desigualdad chilena se está concentrando o dispersando?

## Para profundizar

- Rey, S. & Le Gallo, J. (2009). Spatial analysis of economic convergence. *Palgrave Handbook of Econometrics*.
- Rodríguez-Pose, A. (2018). The revenge of the places that don't matter. *Cambridge Journal of Regions, Economy and Society*, 11(1).